# Week-10

## 230968334- Ashrith Reddy- DSEB- 48

Exercise 1: An autonomous delivery robot must navigate a grid-based
warehouse to reach a goal location while avoiding obstacles.
The environment is represented as a grid:
 Each cell = a state
 Actions = {Up, Down, Left, Right}
 Some cells contain obstacles (cannot enter)
 One cell is the goal state
1. Formulate the problem as an RL problem by defining:
 States
 Actions
 Reward function
 Policy
2. Implement Q-Learning:
 Initialize Q-table
 Use ε-greedy exploration
 Update rule:
Q(s,a) = Q(s,a) + α [r + γ max Q(s',a') − Q(s,a)]
3. Train the agent over multiple episodes. 

In [1]:
import numpy as np
import random

# Grid setup
grid_size = 5
goal = (4, 4)
obstacles = [(1,1), (2,2), (3,1)]

actions = ['up', 'down', 'left', 'right']
Q = {}

# Initialize Q-table
for i in range(grid_size):
    for j in range(grid_size):
        Q[(i,j)] = {a: 0 for a in actions}

# Parameters
alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 500

def step(state, action):
    i, j = state
    
    if action == 'up': i -= 1
    elif action == 'down': i += 1
    elif action == 'left': j -= 1
    elif action == 'right': j += 1

    new_state = (i, j)

    if i < 0 or j < 0 or i >= grid_size or j >= grid_size:
        return state, -10

    if new_state in obstacles:
        return state, -100

    if new_state == goal:
        return new_state, 100

    return new_state, -1

# Training
for _ in range(episodes):
    state = (0,0)

    while state != goal:
        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            action = max(Q[state], key=Q[state].get)

        next_state, reward = step(state, action)

        Q[state][action] += alpha * (
            reward + gamma * max(Q[next_state].values()) - Q[state][action]
        )

        state = next_state

print("Trained Q-values for start:", Q[(0,0)])

Trained Q-values for start: {'up': 24.431768340213804, 'down': 26.967467218592102, 'left': 26.633409539809463, 'right': 42.612658999999276}


Exercise 2: A company wants to choose the best advertisement to show
users. Each ad has an unknown probability of being clicked.
This is modeled as a k-armed bandit problem.
1. Implement:
o ε-greedy strategy
o Upper Confidence Bound (UCB)
2. Simulate:
o 5–10 ads (arms)
o Each with different reward probabilities
3. Track:
o Total reward over time
o Number of times each ad is selected

In [3]:
import numpy as np

k = 5
probs = [0.1, 0.5, 0.3, 0.7, 0.2]

# ε-greedy
epsilon = 0.1
Q = [0]*k
N = [0]*k
total_reward = 0

for t in range(1, 1000):
    if np.random.rand() < epsilon:
        a = np.random.randint(k)
    else:
        a = np.argmax(Q)

    reward = 1 if np.random.rand() < probs[a] else 0

    N[a] += 1
    Q[a] += (reward - Q[a]) / N[a]
    total_reward += reward

print("ε-greedy total reward:", total_reward)

# UCB
Q = [0]*k
N = [0]*k
total_reward = 0

for t in range(1, 1000):
    ucb = [Q[i] + np.sqrt(np.log(t)/(N[i]+1e-5)) for i in range(k)]
    a = np.argmax(ucb)

    reward = 1 if np.random.rand() < probs[a] else 0

    N[a] += 1
    Q[a] += (reward - Q[a]) / N[a]
    total_reward += reward

print("UCB total reward:", total_reward)

ε-greedy total reward: 650
UCB total reward: 665


Exercise 3: A system must balance a pole on a moving cart. The goal is
to keep the pole upright as long as possible.
Use an environment (e.g., OpenAI Gym or custom simulation)
1. Define:
 State (position, velocity, angle, angular velocity)
 Actions (move left/right)
2.Implement:
 Q-Learning (discretized state) OR
 Deep Q-Network (DQN)
3.Train the agent and plot:
 Episode reward vs episodes

In [7]:
import gymnasium as gym
import numpy as np


env = gym.make("CartPole-v1")

# Discretization
bins = [10, 10, 10, 10]
Q = np.zeros(bins + [2])

def discretize(obs):
    upper = [2.4, 3.0, 0.5, 2.0]
    lower = [-2.4, -3.0, -0.5, -2.0]

    ratios = [(obs[i] - lower[i]) / (upper[i] - lower[i]) for i in range(4)]

    new_state = []
    for i in range(4):
        val = int(ratios[i] * (bins[i] - 1))
        val = max(0, min(bins[i] - 1, val))   # 🔥 FIX HERE (CLIPPING)
        new_state.append(val)

    return tuple(new_state)

alpha = 0.1
gamma = 0.95
epsilon = 0.1

rewards = []

for ep in range(500):
    obs = env.reset()[0]
    state = discretize(obs)
    total = 0

    done = False
    while not done:
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])

        next_obs, reward, done, _, _ = env.step(action)
        next_state = discretize(next_obs)

        Q[state][action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state][action]
        )

        state = next_state
        total += reward

    rewards.append(total)

print("Training complete")

Training complete


Exercise 4: Design an intelligent traffic signal system that minimizes
congestion at an intersection.
1. Define:
o States: number of vehicles in each lane
o Actions: which signal to turn green
o Reward: negative of total waiting time
2. Implement:
o Q-Learning agent
3. Simulate traffic flow for multiple time steps
4. Evaluate:
o Average waiting time
o Throughput

In [9]:
import random

states = [(i,j) for i in range(10) for j in range(10)]
actions = ['NS_green', 'EW_green']

Q = {s: {a:0 for a in actions} for s in states}

alpha = 0.1
gamma = 0.9
epsilon = 0.1

def step(state, action):
    ns, ew = state

    if action == 'NS_green':
        ns = max(0, ns - random.randint(1,3))
        ew += random.randint(0,2)
    else:
        ew = max(0, ew - random.randint(1,3))
        ns += random.randint(0,2)

    # 🔥 FIX: keep values within [0, 9]
    ns = min(9, ns)
    ew = min(9, ew)

    reward = -(ns + ew)
    return (ns, ew), reward

for _ in range(500):
    state = random.choice(states)

    for _ in range(50):
        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            action = max(Q[state], key=Q[state].get)

        next_state, reward = step(state, action)

        Q[state][action] += alpha * (
            reward + gamma * max(Q[next_state].values()) - Q[state][action]
        )

        state = next_state

print("Traffic model trained")

Traffic model trained


In [ ]:
Exercise 5: A medical diagnosis system must determine the probability of
a disease given observed symptoms using probabilistic reasoning.
Given
 Prior probability:
P(Disease)
 Conditional probabilities:
P(Symptom | Disease)
P(Symptom | ¬Disease)
1. Apply Bayes’ Theorem:
P(Disease | Symptom) = [P(Symptom | Disease) * P(Disease)] /
P(Symptom)
2. Extend to multiple symptoms assuming independence:
P(S1, S2 | Disease)
3. Implement a Python program to:
o Take input probabilities
o Compute posterior probability
4. Classify: 
o Disease present if probability > threshold

In [10]:
# Input
P_D = float(input("P(Disease): "))
P_S_given_D = float(input("P(Symptom | Disease): "))
P_S_given_notD = float(input("P(Symptom | not Disease): "))

# Total probability
P_S = (P_S_given_D * P_D) + (P_S_given_notD * (1 - P_D))

# Bayes
P_D_given_S = (P_S_given_D * P_D) / P_S

print("Posterior Probability:", P_D_given_S)

# Classification
threshold = 0.5
if P_D_given_S > threshold:
    print("Disease Present")
else:
    print("Disease Not Present")

P(Disease):  0.02
P(Symptom | Disease):  0.8
P(Symptom | not Disease):  0.2


Posterior Probability: 0.07547169811320754
Disease Not Present


In [11]:
# Example for 2 symptoms
P_S1_given_D = 0.8
P_S2_given_D = 0.7

P_S1_given_notD = 0.3
P_S2_given_notD = 0.2

# Independence assumption
P_S_given_D = P_S1_given_D * P_S2_given_D
P_S_given_notD = P_S1_given_notD * P_S2_given_notD

P_S = (P_S_given_D * P_D) + (P_S_given_notD * (1 - P_D))
P_D_given_S = (P_S_given_D * P_D) / P_S

print("Posterior with multiple symptoms:", P_D_given_S)

Posterior with multiple symptoms: 0.16
